# Baseline Linear Model

Out-of-sample, walk-forward linear regression predicting forward returns.
All logic lives in `irp.models.baseline`; this notebook loads a dataset, runs, and plots.

**The dataset comes from the Feature Engineering page** (`/features`): build features + a forward-return
label, then **Export parquet**. This notebook loads that export — it does not rebuild the data.

In [14]:
from irp.models import baseline as bl
from sklearn.linear_model import Ridge

# ── parameters ──
EXPORT_PATH = None        # None = most recent /features export; or a path/filename
FEATURE_COLS = None       # None = infer (all numeric cols except Date/Ticker/fwd_ret/label)
MODEL = Ridge(alpha=1.0)
MIN_TRAIN_DATES = 12
TPL = bl.nb_template()

bl.list_exports()         # available datasets from the /features page

,file,modified,mb
0,features_20260601_034025.parquet,2026-06-01 03:40:29.998514,383.37
1,features_20260601_003532.parquet,2026-06-01 00:35:37.334741,430.52


## 1 · Load dataset (from the /features export)

In [15]:
df, features = bl.load_export(EXPORT_PATH, feature_cols=FEATURE_COLS)
print(len(features), 'features:', features)
df.head()

loaded features_20260601_034025.parquet  3,985,529 rows × 48 cols
44 features: ['rsi_14', 'macd_hist', 'macd_norm', 'bb_pct', 'ma7_ma28', 'ma14_ma56', 'gross_margin', 'op_margin', 'net_margin', 'roe', 'roa', 'roic', 'fcf_margin', 'asset_turnover', 'cfo_ni_ratio', 'accruals', 'revenue', 'net_income', 'total_assets', 'total_equity', 'op_cashflow', 'rand', 'rev_growth_1y', 'earn_growth_1y', 'debt_equity', 'net_debt_ebitda', 'interest_coverage', 'piotroski_fscore', 'close', 'close_lag1', 'close_lag2', 'close_lag3', 'close_lag4', 'close_lag5', 'close_lag6', 'close_lag7', 'volume', 'volume_lag1', 'volume_lag2', 'volume_lag3', 'volume_lag4', 'volume_lag5', 'volume_lag6', 'volume_lag7']


,Date,Ticker,rsi_14,macd_hist,macd_norm,bb_pct,ma7_ma28,ma14_ma56,gross_margin,op_margin,...,volume,volume_lag1,volume_lag2,volume_lag3,volume_lag4,volume_lag5,volume_lag6,volume_lag7,fwd_ret,label
0,2015-01-02,A,47.245697,0.011600,0.000516,0.450383,0.0,1.0,0.423433,0.037602,...,1529200.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.044104,0.0
1,2015-01-09,A,49.732780,-0.037495,-0.003738,0.550023,0.0,1.0,0.423433,0.037602,...,1644900.0,1529200.0,NaN,NaN,NaN,NaN,NaN,NaN,-0.072006,0.0
2,2015-01-16,A,36.442535,-0.224211,-0.014063,0.004695,0.0,0.0,0.423433,0.037602,...,3004000.0,1644900.0,1529200.0,NaN,NaN,NaN,NaN,NaN,0.028098,1.0
3,2015-01-23,A,43.894203,-0.082961,-0.014590,0.280951,0.0,0.0,0.423433,0.037602,...,1519300.0,3004000.0,1644900.0,1529200.0,NaN,NaN,NaN,NaN,0.033944,1.0
4,2015-01-30,A,39.544449,-0.063093,-0.015957,0.141431,0.0,0.0,0.423433,0.037602,...,3054300.0,1519300.0,3004000.0,1644900.0,1529200.0,NaN,NaN,NaN,0.104963,2.0


## 2 · Walk-forward backtest
Expanding window: at each date, fit on the past, predict the current cross-section (no look-ahead).

In [16]:
res = bl.walk_forward_linear(df, features, model=MODEL, min_train_dates=MIN_TRAIN_DATES)
_ = bl.summary(res)

/mnt/Dev/active_python_projects/investment_research_platform/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 4.037894413245108e-19.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/mnt/Dev/active_python_projects/investment_research_platform/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.705801124379122e-19.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/mnt/Dev/active_python_projects/investment_research_platform/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.242797820856693e-19.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/mnt/Dev/active_python_projects/investment_research_platform/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:

mean_IC       0.015851
ICIR          0.195527
R2_oos       -4.157686
n_dates     527.000000
n_preds  673202.000000
topQ_x        2.956667
botQ_x        0.552390
LS_x          5.352498


## 3 · Visualize

In [17]:
bl.plot_quintiles(res, TPL)

In [6]:
bl.plot_ic(res, TPL)

In [7]:
bl.plot_coefs(res, TPL)

In [19]:
bl.plot_pred_vs_actual(res, template=TPL)

In [18]:
res.predictions.head()


,Date,Ticker,fwd_ret,pred
0,2015-03-27,A,0.044484,0.003958
1,2015-03-27,AAL,-0.088704,0.060343
2,2015-03-27,AAOI,0.112462,0.010342
3,2015-03-27,AAP,-0.010558,-0.002978
4,2015-03-27,AAT,-0.044642,0.001137
